# Experimentos 6 y 7 — Análisis Cinemático de Fútbol
**Exp 6:** OC-SORT + ReID (DeepOC-SORT + OSNet) — re-identificación por apariencia  
**Exp 7:** Detector YOLO fine-tuned en dominio fútbol (pseudo-labels) + DeepOC-SORT + ReID

Ejecutar en Google Colab con GPU (Runtime → Change runtime type → T4 GPU)

## Celda 1 — Montar Drive e instalar dependencias

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Instalar dependencias si no están disponibles
!pip install ultralytics pyyaml scikit-learn pandas psutil -q
print('✓ Dependencias listas')

## Celda 2 — Configuración de rutas (ajustar PROJECT_ROOT si es necesario)

In [ ]:
import os, sys, json

# ── AJUSTAR AQUÍ si la carpeta del proyecto tiene otro nombre en tu Drive ──
PROJECT_ROOT = '/content/drive/MyDrive/distancia_futbol_CV'
RESULTS_DIR  = '/content/drive/MyDrive/resultados'
# ──────────────────────────────────────────────────────────────────────────

VIDEO_PATH    = os.path.join(PROJECT_ROOT, 'data', 'raw', 'video.mp4')
PROCESSED_DIR = os.path.join(PROJECT_ROOT, 'data', 'processed')
MODELS_DIR    = os.path.join(PROJECT_ROOT, 'models', 'fine_tuned')

os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)

sys.path.insert(0, PROJECT_ROOT)

assert os.path.exists(VIDEO_PATH), f'Video no encontrado: {VIDEO_PATH}'
assert os.path.exists(PROCESSED_DIR), f'No existe data/processed: {PROCESSED_DIR}'
print('PROJECT_ROOT:', PROJECT_ROOT)
print('VIDEO_PATH:  ', VIDEO_PATH, '→ existe:', os.path.exists(VIDEO_PATH))
print('RESULTS_DIR: ', RESULTS_DIR)

## Celda 3 — Importar módulos del proyecto

In [ ]:
from src.features.projection import calibrate_scale_factor
from src.models.detector import create_deepocsort_yaml
from src.models.tracker import run_controlled_experiment
print('✓ Módulos importados')

## Celda 4 — Cargar resultados previos (Baseline + Exp 1–5)

In [ ]:
all_results = {}

# Intentar cargar desde RESULTS_DIR primero, luego desde PROCESSED_DIR
def load_metrics(key, filename):
    for folder in [RESULTS_DIR, PROCESSED_DIR]:
        path = os.path.join(folder, filename)
        if os.path.exists(path):
            with open(path) as f:
                data = json.load(f)
            print(f'  ✓ {key} cargado desde {folder}')
            return data
    print(f'  ⚠ {key} no encontrado ({filename})')
    return None

# Cargar baseline y experimentos 1-5
entries = [
    ('baseline', 'baseline_metrics.json'),
    ('exp1',     'exp1_metrics.json'),
    ('exp2',     'exp2_metrics.json'),
    ('exp3',     'exp3_metrics.json'),
    ('exp4',     'exp4_metrics.json'),
    ('exp5',     'exp5_metrics.json'),
]

print('Cargando resultados previos...')
for key, fname in entries:
    m = load_metrics(key, fname)
    if m:
        all_results[key] = m

print(f'\nExperimentos cargados: {list(all_results.keys())}')

## Celda 5 — Configuración base compartida (scale factor + polígono)

In [ ]:
# Calibrar scale factor con referencias FIFA (resolución 1364x764)
ref_points_px = [
    ((335, 265), (1020, 265)),  # ancho área penal: 40.32 m → 685 px
    ((575, 345), (855, 345)),   # diámetro círculo central: 18.30 m → 280 px
]
real_distances_m = [40.32, 18.30]
calibrated_scale = calibrate_scale_factor(ref_points_px, real_distances_m)
print(f'Scale factor calibrado: {calibrated_scale:.4f} m/px')

# Polígono de cancha: excluye graderías superiores
field_polygon = [(60, 125), (1300, 125), (1364, 600), (1364, 764), (0, 764), (0, 600)]

## Celda 6 — Experimento 6: DeepOC-SORT + YOLO11s + ReID

**Componente nuevo:** Re-identificación por apariencia (OSNet pretrained en MSMT17)  
**Hipótesis:** Reducir fragmentación residual (~1.7×) al reconocer jugadoras que reaparecen después de oclusiones largas, sin necesidad de fine-tuning del detector.

In [ ]:
# Generar YAML de DeepOC-SORT (OC-SORT + ReID por apariencia)
deepocsort_yaml = create_deepocsort_yaml(
    output_path=os.path.join(MODELS_DIR, 'deepocsort_tuned.yaml'),
    track_high_thresh=0.25,
    track_low_thresh=0.10,
    new_track_thresh=0.60,
    track_buffer=600,        # mismo que Exp 5
    match_thresh=0.80,
    delta_t=5,
    inertia=0.3,
    use_byte=True,
    reid_model='osnet_x0_25_msmt17.pt',   # descarga automática
    proximity_thresh=0.5,
    appearance_thresh=0.25,
)
print(f'YAML generado: {deepocsort_yaml}')

exp6_config = {
    'exp_name':               'Exp 6: DeepOC-SORT + YOLO11s + ReID',
    'tracker_yaml':           deepocsort_yaml,
    'model_path':             'yolo11s.pt',
    'conf_thresh':            0.10,
    'imgsz':                  1536,
    'smooth_window':          9,
    'apply_kinematic_filter': True,
    'deadband_ms':            1.5,
    'max_speed_ms':           12.0,
    'max_gap_frames':         9,
    'scale_factor':           calibrated_scale,
    'use_perspective_scale':  True,
    'field_polygon':          field_polygon,
    'apply_color_clustering': True,
    'apply_track_merge':      True,
    'merge_max_gap_frames':   450,
    'merge_max_dist_px':      300.0,
    'use_homography_gmc':     True,
    'video_path':             VIDEO_PATH,
    'output_dir':             RESULTS_DIR,
    'save_video':             True,
}

print('Iniciando Exp 6...')
metrics6 = run_controlled_experiment(exp6_config, save_artifacts_prefix='exp6')
all_results['exp6'] = metrics6

tr6 = metrics6.get('tracking', {})
di6 = metrics6.get('distances', {})
print(f"\n=== Exp 6 — Resultados ===")
print(f"IDs únicos:        {tr6.get('n_unique_ids')}  (Exp 5: 38)")
print(f"Fragmentación:     {tr6.get('fragmentation_ratio')}×  (Exp 5: 1.7×)")
print(f"Track median:      {tr6.get('median_track_length_frames')} fr  (Exp 5: 940 fr)")
print(f"Dist./jugadora:    {di6.get('estimated_per_player_m')} m  (Exp 5: 162.5 m)")

## Celda 7 — Preparar datos para fine-tuning (pseudo-labels)

Genera automáticamente etiquetas de entrenamiento a partir de las detecciones de alta confianza (conf ≥ 0.50) del modelo YOLO11s preentrenado. Frames muestreados cada 3 fotogramas para diversidad.

> **Nota:** Si tienes acceso a un dataset de fútbol etiquetado (SoccerNet, Roboflow Football), sustituye la carpeta `pseudo_data/images/train` por esas imágenes y etiquetas para obtener mejores resultados.

In [ ]:
import cv2
import numpy as np
import random
from ultralytics import YOLO

PSEUDO_DIR  = '/content/pseudo_data'
FINETUNE_EPOCHS = 20
PSEUDO_CONF     = 0.50   # umbral alto → pseudo-labels de alta calidad
SAMPLE_RATE     = 3      # 1 de cada 3 frames
MIN_DETECTIONS  = 4      # frames con <4 detecciones se descartan

os.makedirs(f'{PSEUDO_DIR}/images/train', exist_ok=True)
os.makedirs(f'{PSEUDO_DIR}/images/val',   exist_ok=True)
os.makedirs(f'{PSEUDO_DIR}/labels/train', exist_ok=True)
os.makedirs(f'{PSEUDO_DIR}/labels/val',   exist_ok=True)

yolo_base = YOLO('yolo11s.pt')
cap = cv2.VideoCapture(VIDEO_PATH)
frame_idx, saved = 0, 0
samples = []

print('Generando pseudo-labels...')
while True:
    ret, frame = cap.read()
    if not ret:
        break
    if frame_idx % SAMPLE_RATE != 0:
        frame_idx += 1
        continue

    results = yolo_base.predict(frame, conf=PSEUDO_CONF, classes=[0], verbose=False)
    boxes = results[0].boxes

    if len(boxes) >= MIN_DETECTIONS:
        H, W = frame.shape[:2]
        label_lines = []
        for box in boxes:
            x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
            cx = ((x1 + x2) / 2) / W
            cy = ((y1 + y2) / 2) / H
            bw = (x2 - x1) / W
            bh = (y2 - y1) / H
            label_lines.append(f'0 {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}')
        samples.append((frame, label_lines, f'frame_{saved:05d}'))
        saved += 1

    frame_idx += 1

cap.release()
print(f'Frames recolectados: {saved}')

# Split 80/20 train/val (aleatorio con semilla fija)
random.seed(42)
random.shuffle(samples)
split = int(len(samples) * 0.8)
train_s, val_s = samples[:split], samples[split:]

for subset, items in [('train', train_s), ('val', val_s)]:
    for frame_img, labels, name in items:
        cv2.imwrite(f'{PSEUDO_DIR}/images/{subset}/{name}.jpg', frame_img)
        with open(f'{PSEUDO_DIR}/labels/{subset}/{name}.txt', 'w') as f:
            f.write('\n'.join(labels))

print(f'Train: {len(train_s)} | Val: {len(val_s)}')

# Crear dataset YAML para fine-tuning
dataset_yaml = '/content/football_pseudo.yaml'
with open(dataset_yaml, 'w') as f:
    f.write(f'path: {PSEUDO_DIR}\ntrain: images/train\nval: images/val\nnc: 1\nnames: [player]\n')
print(f'Dataset YAML: {dataset_yaml}')

## Celda 8 — Fine-tuning de YOLO11s

Ajusta el detector al dominio específico del video: cámara táctica, ángulo cenital, jugadoras pequeñas en el fondo.

In [ ]:
FINETUNE_MODEL_PATH = os.path.join(MODELS_DIR, 'yolo11s_football.pt')

# Fine-tuning: 20 épocas, imgsz=1280 (balance velocidad/calidad)
model_ft = YOLO('yolo11s.pt')
model_ft.train(
    data=dataset_yaml,
    epochs=FINETUNE_EPOCHS,
    imgsz=1280,
    batch=8,
    lr0=0.001,        # learning rate bajo: fine-tuning sobre pesos preentrenados
    patience=10,      # early stopping si no mejora en 10 épocas
    classes=[0],      # solo persona
    seed=42,
    project='/content/yolo_finetune',
    name='exp7_ft',
    exist_ok=True,
    verbose=True,
)

# Copiar best.pt al directorio de modelos del proyecto
import shutil
best_pt = '/content/yolo_finetune/exp7_ft/weights/best.pt'
shutil.copy(best_pt, FINETUNE_MODEL_PATH)
print(f'✓ Modelo fine-tuned guardado en: {FINETUNE_MODEL_PATH}')

# También guardar en Drive
shutil.copy(best_pt, os.path.join(RESULTS_DIR, 'yolo11s_football.pt'))
print(f'✓ Modelo guardado en Drive: {RESULTS_DIR}/yolo11s_football.pt')

## Celda 9 — Experimento 7: YOLO fine-tuned + DeepOC-SORT + ReID

**Componente nuevo respecto a Exp 6:** Detector YOLO11s ajustado al dominio fútbol  
**Hipótesis:** Mejorar recall de jugadoras del fondo y reducir falsos positivos en bordes del polígono

In [ ]:
exp7_config = {
    'exp_name':               'Exp 7: YOLO fine-tuned + DeepOC-SORT + ReID',
    'tracker_yaml':           deepocsort_yaml,          # mismo que Exp 6
    'model_path':             FINETUNE_MODEL_PATH,       # detector ajustado
    'conf_thresh':            0.10,
    'imgsz':                  1536,
    'smooth_window':          9,
    'apply_kinematic_filter': True,
    'deadband_ms':            1.5,
    'max_speed_ms':           12.0,
    'max_gap_frames':         9,
    'scale_factor':           calibrated_scale,
    'use_perspective_scale':  True,
    'field_polygon':          field_polygon,
    'apply_color_clustering': True,
    'apply_track_merge':      True,
    'merge_max_gap_frames':   450,
    'merge_max_dist_px':      300.0,
    'use_homography_gmc':     True,
    'video_path':             VIDEO_PATH,
    'output_dir':             RESULTS_DIR,
    'save_video':             True,
}

print('Iniciando Exp 7...')
metrics7 = run_controlled_experiment(exp7_config, save_artifacts_prefix='exp7')
all_results['exp7'] = metrics7

tr7 = metrics7.get('tracking', {})
di7 = metrics7.get('distances', {})
print(f"\n=== Exp 7 — Resultados ===")
print(f"IDs únicos:        {tr7.get('n_unique_ids')}  (Exp 6: {all_results['exp6'].get('tracking',{}).get('n_unique_ids')})")
print(f"Fragmentación:     {tr7.get('fragmentation_ratio')}×  (Exp 6: {all_results['exp6'].get('tracking',{}).get('fragmentation_ratio')}×)")
print(f"Track median:      {tr7.get('median_track_length_frames')} fr  (Exp 6: {all_results['exp6'].get('tracking',{}).get('median_track_length_frames')} fr)")
print(f"Dist./jugadora:    {di7.get('estimated_per_player_m')} m  (Exp 6: {all_results['exp6'].get('distances',{}).get('estimated_per_player_m')} m)")

## Celda 10 — Guardar todos los resultados en Drive

In [ ]:
import shutil

# ── 1. ablation_results.json completo (baseline + exp1-7) ─────────────────────
ablation_path = os.path.join(RESULTS_DIR, 'ablation_results_completo.json')
with open(ablation_path, 'w') as f:
    json.dump(all_results, f, indent=2)
print(f'✓ ablation_results_completo.json → {RESULTS_DIR}')

# También actualizar en data/processed/ del proyecto
local_ablation = os.path.join(PROCESSED_DIR, 'ablation_results.json')
with open(local_ablation, 'w') as f:
    json.dump(all_results, f, indent=2)
print(f'✓ ablation_results.json actualizado en data/processed/')

# ── 2. Copiar métricas individuales exp6 y exp7 a Drive (backup) ──────────────
for prefix in ['exp6', 'exp7']:
    for suffix in ['_metrics.json', '_distances.csv', '_frame_data.csv']:
        src = os.path.join(RESULTS_DIR, prefix + suffix)
        if os.path.exists(src):
            print(f'✓ {prefix}{suffix} ya está en Drive')
        else:
            # Por si se guardó en processed/ en vez de Drive
            local = os.path.join(PROCESSED_DIR, prefix + suffix)
            if os.path.exists(local):
                shutil.copy(local, os.path.join(RESULTS_DIR, prefix + suffix))
                print(f'✓ {prefix}{suffix} copiado a Drive')

# ── 3. Listar todos los archivos en RESULTS_DIR ───────────────────────────────
print('\nContenido de', RESULTS_DIR)
for f in sorted(os.listdir(RESULTS_DIR)):
    size_kb = os.path.getsize(os.path.join(RESULTS_DIR, f)) // 1024
    print(f'  {f:<50} {size_kb:>6} KB')

## Celda 11 — Tabla comparativa final (todos los experimentos)

In [ ]:
def fmt(d, *keys, default='N/A', dec=2):
    v = d
    for k in keys:
        v = v.get(k, default) if isinstance(v, dict) else default
    return f'{v:.{dec}f}' if isinstance(v, float) else str(v)

LABELS = {
    'baseline': 'Baseline',
    'exp1': 'Exp 1', 'exp2': 'Exp 2', 'exp3': 'Exp 3',
    'exp4': 'Exp 4', 'exp5': 'Exp 5*',
    'exp6': 'Exp 6', 'exp7': 'Exp 7',
}

keys = [k for k in ['baseline','exp1','exp2','exp3','exp4','exp5','exp6','exp7']
        if k in all_results]

W = 12
header = f'{"Métrica":<30}' + ''.join(f'{LABELS[k]:>{W}}' for k in keys)
print('\n' + '='*len(header))
print('TABLA COMPARATIVA COMPLETA — ESTUDIO DE ABLACIÓN')
print('='*len(header))
print(header)
print('-'*len(header))

rows = [
    ('IDs únicos',          'tracking', 'n_unique_ids'),
    ('Fragmentación (×)',   'tracking', 'fragmentation_ratio'),
    ('ID switches',         'tracking', 'id_switches_estimated'),
    ('Track median (fr)',   'tracking', 'median_track_length_frames'),
    ('Ghost tracks',        'tracking', 'n_ghost_tracks_by_dist'),
    ('Vel. max (m/s)',       'kinematics', 'max_speed_ms'),
    ('% vel. imposible',    'kinematics', 'pct_impossible_speed'),
    ('Var. aceleración',    'smoothness', 'mean_accel_variance'),
    ('Dist./jugadora (m)',  'distances', 'estimated_per_player_m'),
    ('FPS procesamiento',   'computational_cost', 'fps_processing'),
]

for label, cat, metric in rows:
    vals = ''.join(f'{fmt(all_results[k], cat, metric):>{W}}' for k in keys)
    print(f'{label:<30}{vals}')

print('-'*len(header))
# Extrapolación a 90 min
ext_vals = []
for k in keys:
    d = all_results[k]
    dist = d.get('distances', {}).get('estimated_per_player_m', 0)
    frames = d.get('computational_cost', {}).get('total_frames', 1233)
    vfps   = d.get('computational_cost', {}).get('video_fps', 30)
    dur_min = frames / vfps / 60
    km90 = dist / dur_min * 90 / 1000 if dur_min > 0 else 0
    ext_vals.append(f'{km90:.1f}')
print(f'{"Extrap. 90 min (km)":<30}' + ''.join(f'{v:>{W}}' for v in ext_vals))
print('='*len(header))
print('* Exp 5 ejecutado en sesión previa de Colab')